In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from transformers import AutoImageProcessor, AutoModel

/Users/lshahsha/Documents/GitHub/model_tutorial/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Linear probing

Linear probing means: We freeze a pretrained encoder and train only a linear classifier on top of its representations.

We want to know whether the featuers (distributions) that Dino has learnt are enough to be used by a linear classifier to classify never-seen images. We are asking:
> How linearly separable are the representations?

- freeze the encoder so that it becomes a fixed feature extractor
    - note that this is different from putting the model in eval mode. Eval mode changes the behaviour of layers. Specifically Dropout and BatchNorm 
- train the linear classifier
- Performance reflects how good the representations already are

once we load and freeze the model we can use features from different layers to linearly probe. We can also choose to use CLS tokens, global mean (pooling over the tokens), etc. For simplicity, we will be using CLS tokens from each layer. 

Here in this notebook, we will see how we can use the cls tokens of the last layer to linearly probe. We will then use cls tokens from intermediate layers as well. 

In [2]:
model_name = "facebook/dinov2-small"

# lets set the "device" to be either the GPU if available, or the CPU if not. This will allow us to run the code on both GPU and CPU without any changes.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# first get the processor for the specified pretrained model
processor = AutoImageProcessor.from_pretrained(model_name)

# now get the model itself and the pretrained weights
encoder = AutoModel.from_pretrained(model_name)
print(encoder)

# freeze all the weights in the model so that they are not updated during training
for param in encoder.parameters():
    param.requires_grad = False # set requires_grad to False to freeze the weights

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 223/223 [00:00<00:00, 1156.42it/s, Materializing param=layernorm.weight]                                 

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-06,

In [ ]:
# first lets define the datasets
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from torch.utils.data import Subset


def collate_fn(batch):
    # print(type(batch))
    
    images = [item[0] for item in batch]
    labels = torch.tensor([item[1] for item in batch])

    inputs = processor(
        images,
        return_tensors="pt"
    )

    return inputs["pixel_values"], labels


# NOTE: the dataset object you load does have a transform argument. 
# you can pass on the transform object so that the transformation is automatically applied to 
# the downloaded images. 
# but we are using huggingface transformers image processor to do the transformations, 
# and the transformers image processor expects a PIL image as input.
# we do not want to apply the default transformations that come with the CIFAR10 dataset (which converts the image to a tensor),
# but later in training loop, before we pass the image to the model, we will apply the transformations using the transformers image processor.
# so this is not like the way we trained our model in the training notebook


train_dataset = CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=None
)

test_dataset = CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=None
)

# we will only focus on a subset of images for faster training
N = 200  # e.g. only 200 images instead of 50,000

train_subset = Subset(train_dataset, range(N))
test_subset = Subset(test_dataset, range(500))  # smaller test set too



train_loader = DataLoader(train_subset, batch_size=10, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_subset, batch_size=10, shuffle=False, collate_fn=collate_fn)


sample = train_dataset[0]
sample

# for image, label in train_loader:
#     print(type(image))
#     print(label)


/Users/lshahsha/Documents/GitHub/model_tutorial/.venv/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


(<PIL.Image.Image image mode=RGB size=32x32>, 6)

So now we have frozen the weights/params of our dino model. We will next define a simple linear classifier head. During training the paramters/weights of this linear classifier will be updated:

Encoder weights: frozen

Classifier weights: randomly initialized

Optimizer will only update classifier (because we froze the parameters of the Dino encoder)

In [4]:
# define a linear classifier on top of the frozen encoder
class LinearProbe(torch.nn.Module):
    def __init__(self, encoder, embed_dim=384, num_classes=10):
        # there are 10 classes in CIFAR10, so the output dimension of the linear layer should be 10
        super().__init__()
        # get the frozen encoder as a submodule of the linear probe
        self.encoder = encoder
        # define a linear layer that takes the encoder output and maps it to the number of classes
        self.classifier = torch.nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # first pass the input through the frozen encoder to get the image embeddings
        with torch.no_grad(): # disable gradient computation for the encoder since it's frozen
            encoder_outputs = self.encoder(x)
            # the encoder output is a sequence of tokens, we take the CLS token (the first token) as the image embedding
            image_embedding = encoder_outputs.last_hidden_state[:, 0, :] # shape (B, embed_dim)
        # now pass the image embedding through the linear classifier to get the class logits
        logits = self.classifier(image_embedding) # shape (B, num_classes)
        return logits

In [5]:
num_classes = 10
embed_dim = 384  # DINO small

# create the linear probe model by passing the frozen encoder, embedding dimension, and number of classes
model = LinearProbe(
    encoder=encoder,
    embed_dim=embed_dim,
    num_classes=num_classes
)


# now lets confirm that the encoder weights are frozen and the classifier weights are trainable
for name, param in model.named_parameters():
    print(name, param.requires_grad)



encoder.embeddings.cls_token False
encoder.embeddings.mask_token False
encoder.embeddings.position_embeddings False
encoder.embeddings.patch_embeddings.projection.weight False
encoder.embeddings.patch_embeddings.projection.bias False
encoder.encoder.layer.0.norm1.weight False
encoder.encoder.layer.0.norm1.bias False
encoder.encoder.layer.0.attention.attention.query.weight False
encoder.encoder.layer.0.attention.attention.query.bias False
encoder.encoder.layer.0.attention.attention.key.weight False
encoder.encoder.layer.0.attention.attention.key.bias False
encoder.encoder.layer.0.attention.attention.value.weight False
encoder.encoder.layer.0.attention.attention.value.bias False
encoder.encoder.layer.0.attention.output.dense.weight False
encoder.encoder.layer.0.attention.output.dense.bias False
encoder.encoder.layer.0.layer_scale1.lambda1 False
encoder.encoder.layer.0.norm2.weight False
encoder.encoder.layer.0.norm2.bias False
encoder.encoder.layer.0.mlp.fc1.weight False
encoder.encoder.

In [6]:
# now we define the loss and optimizer for training the linear probe
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3) # only optimize the classifier parameters, not the encoder parameters since they are frozen

In [7]:
def train_model(train_loader, model, criterion, optimizer, max_epochs=10):

    
    for epoch in range(1, max_epochs + 1):
        print(f"\nEpoch {epoch}/{max_epochs}")

        model.train()
        total_loss = 0.0
        accuracy_history = []
        loss_history = []
        for image, label in train_loader:
            # image is ALREADY (B, 3, 224, 224)
            image = image.to(device)
            label = label.to(device)

            # forward pass
            logits = model(image)
            loss = criterion(logits, label)

            # backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            loss_history.append(loss.item())

            preds = logits.argmax(dim=1)
            accuracy = (preds == label).float().mean().item()
            accuracy_history.append(accuracy)

        avg_loss = total_loss / len(train_loader)
        avg_accuracy = sum(accuracy_history) / len(accuracy_history)

        print(f"Train Loss: {avg_loss:.4f}, Train Accuracy: {avg_accuracy:.4f}")

    return loss_history, accuracy_history

def evaluate_model(test_loader, model, criterion, processor):

    # first put the model in evaluation mode
    # note that in evaluation mode, Dropout and BatchNorm layers will be turned off, which is important for getting accurate evaluation results
    model.eval()

    total_loss = 0.0
    accuracy_history = []
    with torch.no_grad(): # disable gradient computation during evaluation since we are not training
        for image, label in test_loader:
            # move the image and label to the same device as the model
            image = image.to(device)
            label = label.to(device)

            # apply the transformations to image using the transformers image processor
            # image = processor(images=image, return_tensors="pt")["pixel_values"]
            logits = model(image)

            # calculate the loss
            loss = criterion(logits, label)
            total_loss += loss.item()

            # get the predicted class by taking the argmax of the logits
            _, predicted = torch.max(logits, 1)

            # calculate the accuracy by comparing the predicted class with the true label
            accuracy = (predicted == label).float().mean().item()
            accuracy_history.append(accuracy)

    # get the final average loss and accuracy for the evaluation
    avg_loss = total_loss / len(test_loader)
    avg_accuracy = sum(accuracy_history) / len(accuracy_history)
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {avg_accuracy:.4f}")  
    return avg_loss, avg_accuracy


In [8]:
loss_history, accuracy_history = train_model(train_loader, model, criterion, optimizer, max_epochs=2)


Epoch 1/2
Train Loss: 1.9083, Train Accuracy: 0.3850

Epoch 2/2
Train Loss: 0.4732, Train Accuracy: 0.8650


In [9]:
avg_loss, avg_acc = evaluate_model(test_loader, model, criterion, processor)

Test Loss: 0.6388, Test Accuracy: 0.8060
